# Calculo de D = C^T d C, S = D # G e s = A^T S A

Este notebook calcula as etapas da transformacao uma saida por vez.

Primeiro calcula-se `D = C^T d C`. Para cada coluna `j` de `C`:

1. Calcula-se a coluna intermediaria `D'[:, j] = d C[:, j]`, multiplicando cada linha de `d` pela coluna `j` de `C`.
2. Essa coluna intermediaria e entao multiplicada pelas linhas de `C^T`, gerando uma coluna de `D`.

Depois calcula-se `S = D # G`, onde `#` e multiplicacao ponto a ponto.

Por fim calcula-se `s = A^T S A`, repetindo o mesmo processo coluna por coluna.

In [1]:
import numpy as np
from fractions import Fraction

C = np.array([
    [-1, 0,  0,  0],
    [ 0, 1, -1, -1],
    [ 1, 1,  1,  0],
    [ 0, 0,  0,  1],
], dtype=int)

d = np.array([
    [ 0,  1,  2,  3],
    [ 4,  5,  6,  7],
    [ 8,  9, 10, 11],
    [12, 13, 14, 15],
], dtype=int)

print("C =")
print(C)
print("\nd =")
print(d)
print("\nC^T =")
print(C.T)


C =
[[-1  0  0  0]
 [ 0  1 -1 -1]
 [ 1  1  1  0]
 [ 0  0  0  1]]

d =
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]

C^T =
[[-1  0  1  0]
 [ 0  1  1  0]
 [ 0 -1  1  0]
 [ 0 -1  0  1]]


In [2]:
def dot_expression(a, b):
    return " + ".join(f"({x})*({y})" for x, y in zip(a, b))


def print_column_step(j, intermediate_col, output_col):
    print(f"Coluna {j} de C:")
    print(C[:, j])
    print(f"\n1) Calculando D'[:, {j}] = d C[:, {j}]")

    for row in range(d.shape[0]):
        expr = dot_expression(d[row, :], C[:, j])
        print(f"D'[{row}, {j}] = {expr} = {intermediate_col[row]}")

    print(f"\nD'[:, {j}] = {intermediate_col.tolist()}")
    print(f"\n2) Calculando D[:, {j}] = C^T D'[:, {j}]")

    for row in range(C.T.shape[0]):
        expr = dot_expression(C.T[row, :], intermediate_col)
        print(f"D[{row}, {j}] = {expr} = {output_col[row]}")

    print(f"\nD[:, {j}] = {output_col.tolist()}")


## Calculo de D = C^T d C

A matriz `D' = d C` nao precisa ser formada inteira de uma vez. Abaixo, cada coluna de `D'` e calculada separadamente e imediatamente usada para produzir a coluna correspondente de `D`.

In [3]:
D = np.zeros((4, 4), dtype=int)
intermediate_columns = []

for j in range(C.shape[1]):
    intermediate_col = d @ C[:, j]
    output_col = C.T @ intermediate_col

    D[:, j] = output_col
    intermediate_columns.append(intermediate_col)

    print("=" * 70)
    print_column_step(j, intermediate_col, output_col)
    print()


Coluna 0 de C:
[-1  0  1  0]

1) Calculando D'[:, 0] = d C[:, 0]
D'[0, 0] = (0)*(-1) + (1)*(0) + (2)*(1) + (3)*(0) = 2
D'[1, 0] = (4)*(-1) + (5)*(0) + (6)*(1) + (7)*(0) = 2
D'[2, 0] = (8)*(-1) + (9)*(0) + (10)*(1) + (11)*(0) = 2
D'[3, 0] = (12)*(-1) + (13)*(0) + (14)*(1) + (15)*(0) = 2

D'[:, 0] = [2, 2, 2, 2]

2) Calculando D[:, 0] = C^T D'[:, 0]
D[0, 0] = (-1)*(2) + (0)*(2) + (1)*(2) + (0)*(2) = 0
D[1, 0] = (0)*(2) + (1)*(2) + (1)*(2) + (0)*(2) = 4
D[2, 0] = (0)*(2) + (-1)*(2) + (1)*(2) + (0)*(2) = 0
D[3, 0] = (0)*(2) + (-1)*(2) + (0)*(2) + (1)*(2) = 0

D[:, 0] = [0, 4, 0, 0]

Coluna 1 de C:
[0 1 1 0]

1) Calculando D'[:, 1] = d C[:, 1]
D'[0, 1] = (0)*(0) + (1)*(1) + (2)*(1) + (3)*(0) = 3
D'[1, 1] = (4)*(0) + (5)*(1) + (6)*(1) + (7)*(0) = 11
D'[2, 1] = (8)*(0) + (9)*(1) + (10)*(1) + (11)*(0) = 19
D'[3, 1] = (12)*(0) + (13)*(1) + (14)*(1) + (15)*(0) = 27

D'[:, 1] = [3, 11, 19, 27]

2) Calculando D[:, 1] = C^T D'[:, 1]
D[0, 1] = (-1)*(3) + (0)*(11) + (1)*(19) + (0)*(27) = 16
D[1, 1] =

## Resultado de D

In [4]:
D_prime = np.column_stack(intermediate_columns)
D_direct = C.T @ d @ C

print("D' = d C =")
print(D_prime)
print("\nD = C^T D' = C^T d C =")
print(D)

assert np.array_equal(D, D_direct)
print("\nVerificacao: D e igual a C^T @ d @ C.")


D' = d C =
[[ 2  3  1  2]
 [ 2 11  1  2]
 [ 2 19  1  2]
 [ 2 27  1  2]]

D = C^T D' = C^T d C =
[[ 0 16  0  0]
 [ 4 30  2  4]
 [ 0  8  0  0]
 [ 0 16  0  0]]

Verificacao: D e igual a C^T @ d @ C.


## Multiplicacao ponto a ponto: S = D # G

O operador `#` representa multiplicacao elemento a elemento. Assim, cada valor de `S` e calculado como `S[i, j] = D[i, j] * G[i, j]`.

In [5]:
G = np.array([
    [Fraction(0),    Fraction(-3, 2), Fraction(-1, 2), Fraction(-2)],
    [Fraction(-9, 2), Fraction(9),     Fraction(3),     Fraction(15, 2)],
    [Fraction(-3, 2), Fraction(3),     Fraction(1),     Fraction(5, 2)],
    [Fraction(-6),    Fraction(21, 2), Fraction(7, 2),  Fraction(8)],
], dtype=object)

S = D.astype(object) * G

print("G =")
print(G)
print("\nS = D # G =")

for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        print(f"S[{i}, {j}] = D[{i}, {j}] * G[{i}, {j}] = ({D[i, j]})*({G[i, j]}) = {S[i, j]}")

print("\nS =")
print(S)


G =
[[Fraction(0, 1) Fraction(-3, 2) Fraction(-1, 2) Fraction(-2, 1)]
 [Fraction(-9, 2) Fraction(9, 1) Fraction(3, 1) Fraction(15, 2)]
 [Fraction(-3, 2) Fraction(3, 1) Fraction(1, 1) Fraction(5, 2)]
 [Fraction(-6, 1) Fraction(21, 2) Fraction(7, 2) Fraction(8, 1)]]

S = D # G =
S[0, 0] = D[0, 0] * G[0, 0] = (0)*(0) = 0
S[0, 1] = D[0, 1] * G[0, 1] = (16)*(-3/2) = -24
S[0, 2] = D[0, 2] * G[0, 2] = (0)*(-1/2) = 0
S[0, 3] = D[0, 3] * G[0, 3] = (0)*(-2) = 0
S[1, 0] = D[1, 0] * G[1, 0] = (4)*(-9/2) = -18
S[1, 1] = D[1, 1] * G[1, 1] = (30)*(9) = 270
S[1, 2] = D[1, 2] * G[1, 2] = (2)*(3) = 6
S[1, 3] = D[1, 3] * G[1, 3] = (4)*(15/2) = 30
S[2, 0] = D[2, 0] * G[2, 0] = (0)*(-3/2) = 0
S[2, 1] = D[2, 1] * G[2, 1] = (8)*(3) = 24
S[2, 2] = D[2, 2] * G[2, 2] = (0)*(1) = 0
S[2, 3] = D[2, 3] * G[2, 3] = (0)*(5/2) = 0
S[3, 0] = D[3, 0] * G[3, 0] = (0)*(-6) = 0
S[3, 1] = D[3, 1] * G[3, 1] = (16)*(21/2) = 168
S[3, 2] = D[3, 2] * G[3, 2] = (0)*(7/2) = 0
S[3, 3] = D[3, 3] * G[3, 3] = (0)*(8) = 0

S =
[[Fracti

## Calculo de s = A^T S A

Agora o mesmo processo e repetido para `s = A^T S A`. Para cada coluna `j` de `A`:

1. Calcula-se a coluna intermediaria `s'[:, j] = S A[:, j]`, multiplicando cada linha de `S` pela coluna `j` de `A`.
2. Essa coluna intermediaria e multiplicada pelas linhas de `A^T`, gerando a coluna correspondente de `s`.

In [6]:
A = np.array([
    [1,  0],
    [1,  1],
    [1, -1],
    [0,  1],
], dtype=object)

print("A =")
print(A)
print("\nA^T =")
print(A.T)


A =
[[1 0]
 [1 1]
 [1 -1]
 [0 1]]

A^T =
[[1 1 1 0]
 [0 1 -1 1]]


In [7]:
def print_s_column_step(j, intermediate_col, output_col):
    print(f"Coluna {j} de A:")
    print(A[:, j])
    print(f"\n1) Calculando s'[:, {j}] = S A[:, {j}]")

    for row in range(S.shape[0]):
        expr = dot_expression(S[row, :], A[:, j])
        print(f"s'[{row}, {j}] = {expr} = {intermediate_col[row]}")

    print(f"\ns'[:, {j}] = {intermediate_col.tolist()}")
    print(f"\n2) Calculando s[:, {j}] = A^T s'[:, {j}]")

    for row in range(A.T.shape[0]):
        expr = dot_expression(A.T[row, :], intermediate_col)
        print(f"s[{row}, {j}] = {expr} = {output_col[row]}")

    print(f"\ns[:, {j}] = {output_col.tolist()}")


In [8]:
s = np.zeros((2, 2), dtype=object)
s_intermediate_columns = []

for j in range(A.shape[1]):
    intermediate_col = S @ A[:, j]
    output_col = A.T @ intermediate_col

    s[:, j] = output_col
    s_intermediate_columns.append(intermediate_col)

    print("=" * 70)
    print_s_column_step(j, intermediate_col, output_col)
    print()


Coluna 0 de A:
[1 1 1 0]

1) Calculando s'[:, 0] = S A[:, 0]
s'[0, 0] = (0)*(1) + (-24)*(1) + (0)*(1) + (0)*(0) = -24
s'[1, 0] = (-18)*(1) + (270)*(1) + (6)*(1) + (30)*(0) = 258
s'[2, 0] = (0)*(1) + (24)*(1) + (0)*(1) + (0)*(0) = 24
s'[3, 0] = (0)*(1) + (168)*(1) + (0)*(1) + (0)*(0) = 168

s'[:, 0] = [Fraction(-24, 1), Fraction(258, 1), Fraction(24, 1), Fraction(168, 1)]

2) Calculando s[:, 0] = A^T s'[:, 0]
s[0, 0] = (1)*(-24) + (1)*(258) + (1)*(24) + (0)*(168) = 258
s[1, 0] = (0)*(-24) + (1)*(258) + (-1)*(24) + (1)*(168) = 402

s[:, 0] = [Fraction(258, 1), Fraction(402, 1)]

Coluna 1 de A:
[0 1 -1 1]

1) Calculando s'[:, 1] = S A[:, 1]
s'[0, 1] = (0)*(0) + (-24)*(1) + (0)*(-1) + (0)*(1) = -24
s'[1, 1] = (-18)*(0) + (270)*(1) + (6)*(-1) + (30)*(1) = 294
s'[2, 1] = (0)*(0) + (24)*(1) + (0)*(-1) + (0)*(1) = 24
s'[3, 1] = (0)*(0) + (168)*(1) + (0)*(-1) + (0)*(1) = 168

s'[:, 1] = [Fraction(-24, 1), Fraction(294, 1), Fraction(24, 1), Fraction(168, 1)]

2) Calculando s[:, 1] = A^T s'[:, 1]

## Resultado final de s

In [9]:
s_prime = np.column_stack(s_intermediate_columns)
s_direct = A.T @ S @ A

print("s' = S A =")
print(s_prime)
print("\ns = A^T s' = A^T S A =")
print(s)

assert np.array_equal(s, s_direct)
print("\nVerificacao: s e igual a A.T @ S @ A.")


s' = S A =
[[Fraction(-24, 1) Fraction(-24, 1)]
 [Fraction(258, 1) Fraction(294, 1)]
 [Fraction(24, 1) Fraction(24, 1)]
 [Fraction(168, 1) Fraction(168, 1)]]

s = A^T s' = A^T S A =
[[Fraction(258, 1) Fraction(294, 1)]
 [Fraction(402, 1) Fraction(438, 1)]]

Verificacao: s e igual a A.T @ S @ A.


In [12]:

s 

array([[Fraction(258, 1), Fraction(294, 1)],
       [Fraction(402, 1), Fraction(438, 1)]], dtype=object)